## Import libraries

In [22]:
import SimpleITK as sitk # type: ignore
import os
import numpy as np # type: ignore
import pandas as pd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import shutil # type: ignore
import seaborn as sns # type: ignore
from skimage.metrics import peak_signal_noise_ratio as psnr # type: ignore
from skimage.metrics import structural_similarity as ssim # type: ignore
from sklearn.preprocessing import MinMaxScaler # type: ignore
from sklearn.preprocessing import StandardScaler # type: ignore
from sklearn.preprocessing import MaxAbsScaler # type: ignore
from sklearn.preprocessing import RobustScaler # type: ignore
from skimage.feature import graycomatrix, graycoprops
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

## Load Datasets

In [4]:
df_adc = pd.read_csv("../Part3/df_adc.csv").set_index("Study Instance UID")
df_hbv = pd.read_csv("../Part3/df_hbv.csv").set_index("Study Instance_UID")
df_t2w = pd.read_csv("../Part3/df_t2w.csv").set_index("Study Instance UID")
glcm_df = pd.read_csv("../Part3/glcm_df.csv")

In [15]:
glcm_df.drop("Unnamed: 0", axis=1)

,slice,type,degree,contrast,correlation,homogeneity,patient ID
0,5,t2w,0.000000,607.837224,0.563103,0.855344,1000001
1,5,t2w,0.785398,950.122353,0.171183,0.839276,1000001
2,5,t2w,1.570796,694.328554,0.473714,0.829771,1000001
3,5,t2w,2.356194,953.113572,0.169284,0.840279,1000001
4,6,t2w,0.000000,499.329075,0.602381,0.855344,1000001
...,...,...,...,...,...,...,...
27055,14,hbv,2.356194,0.605306,0.773803,0.840279,1001497
27056,15,hbv,0.000000,0.329795,0.864513,0.855344,1001497
27057,15,hbv,0.785398,0.581730,0.759949,0.839276,1001497
27058,15,hbv,1.570796,0.422105,0.826275,0.829771,1001497


In [7]:
df_t2w

,Manufacturer,Manufacturer's Model Name,Patient's Age,PROSTATE_VOLUME_REPORT,PSAD_REPORT,PSA_REPORT,path,label,path_edged
Study Instance UID,,,,,,,,,
1000001,SIEMENS,Skyra,64,102.0,0.09,8.70,Dataset_v3/10001_1000001_t2w.mha,0,edge_detected/10001_1000001_t2w.mha
1000006,SIEMENS,Skyra,73,27.0,0.23,6.20,Dataset_v3/10006_1000006_t2w.mha,0,edge_detected/10006_1000006_t2w.mha
1000020,SIEMENS,Skyra,43,47.0,0.11,4.60,Dataset_v3/10020_1000020_t2w.mha,0,edge_detected/10020_1000020_t2w.mha
1000023,SIEMENS,TrioTim,62,37.0,0.03,1.50,Dataset_v3/10023_1000023_t2w.mha,0,edge_detected/10023_1000023_t2w.mha
1000029,SIEMENS,Skyra,64,46.0,0.20,8.90,Dataset_v3/10029_1000029_t2w.mha,1,edge_detected/10029_1000029_t2w.mha
...,...,...,...,...,...,...,...,...,...
1001475,SIEMENS,Skyra,71,44.0,0.14,5.58,Dataset_v3/11451_1001475_t2w.mha,1,edge_detected/11451_1001475_t2w.mha
1001491,SIEMENS,Skyra,61,103.0,0.05,5.40,Dataset_v3/11467_1001491_t2w.mha,0,edge_detected/11467_1001491_t2w.mha
1001493,SIEMENS,Skyra,49,34.0,0.13,4.30,Dataset_v3/11469_1001493_t2w.mha,0,edge_detected/11469_1001493_t2w.mha


## Build CNN Models

### Hybrid model (slicing)

In [23]:
inputs = keras.Input(shape=(256, 256, 21), name="image_input")
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
    
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)

glcm_input = keras.Input(shape=(4,), name="glcm_input")
y = layers.Dense(32, activation='relu')(glcm_input)
y = layers.Dense(32, activation='relu')(y)

combined = keras.layers.concatenate([x, y])

z = layers.Dense(64, activation='relu')(combined)
z = layers.Dropout(0.5)(z)
outputs = layers.Dense(1, activation='sigmoid')(z)

model = keras.Model(inputs=inputs, outputs=outputs)

In [24]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 21)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 256, 256,  │      6,080 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 128, 128,  │          0 │ conv2d_15[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 128, 128,  │     18,496 │ max_pooling2d_15… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_16    │ (None, 64, 64,    │          0 │ conv2d_16[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_16… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_17    │ (None, 32, 32,    │          0 │ conv2d_17[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glcm_input          │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 131072)    │          0 │ max_pooling2d_17… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 32)        │        160 │ glcm_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 128)       │ 16,777,344 │ flatten_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 32)        │      1,056 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 160)       │          0 │ dense_15[0][0],   │
│ (Concatenate)       │                   │            │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 64)        │     10,304 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 1)         │         65 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,887,361 (64.42 MB)

 Trainable params: 16,887,361 (64.42 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])